# Churn Scoring Platform — EDA и анализ ликажа

**Датасет:** IBM Telco Customer Churn (7043 клиента, 1 квартал).
Ноутбук решает две задачи: (1) разведочный анализ факторов оттока и
(2) явную защиту от утечки целевой переменной (data leakage) — показать,
почему из сырых 33 колонок часть исключена ещё до обучения.

## 1. Цель и данные

Задача — предсказать **отток абонента**. В этих данных *уходящий клиент* —
это клиент, **расторгнувший контракт с оператором в отчётном квартале**
(поле `Churn Value` из IBM-сэмпла: 1 — ушёл, 0 — остался). Основной источник
для EDA — очищенные данные `data/interim/telco_clean.parquet` (snake_case,
таргет `churn`); сырой файл `data/raw/Telco_customer_churn.xlsx` используется
только в секции анализа ликажа. Все вычисления детерминированы, `seed = 42`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import roc_auc_score

from churn.features import build_features, ALL_FEATURES, CATEGORICAL_FEATURES, NUMERIC_FEATURES
from churn.data import TARGET, LEAKY_COLUMNS

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ноутбук может исполняться из notebooks/, поэтому ищем корень репозитория
ROOT = Path.cwd()
while not (ROOT / "data" / "interim").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# --- единая палитра проекта -------------------------------------------------
BLUE, ORANGE, GRAY, INK = "#2563eb", "#ea580c", "#6b7280", "#111827"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.labelsize": 10, "axes.edgecolor": GRAY, "axes.linewidth": 0.8,
})


def style_ax(ax, value_axis="y"):
    """Убрать верхнюю/правую рамки, лёгкая сетка по оси значений."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis="x" if value_axis == "x" else "y", alpha=0.3, linewidth=0.7)
    ax.tick_params(length=0)


def label_bars(ax, orient="v", fmt="{:.1f}%"):
    """Прямые подписи значений на барах."""
    for p in ax.patches:
        if orient == "v":
            v = p.get_height()
            ax.annotate(fmt.format(v), (p.get_x() + p.get_width() / 2, v),
                        ha="center", va="bottom", fontsize=9, color=INK,
                        xytext=(0, 2), textcoords="offset points")
        else:
            v = p.get_width()
            ax.annotate(fmt.format(v), (v, p.get_y() + p.get_height() / 2),
                        ha="left", va="center", fontsize=9, color=INK,
                        xytext=(3, 0), textcoords="offset points")


def save(fig, name):
    fig.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")
    print("saved:", name)


clean = pd.read_parquet(ROOT / "data" / "interim" / "telco_clean.parquet")
feat = build_features(clean)
print("clean:", clean.shape, "| featured:", feat.shape)

## 2. Таргет: распределение классов

Задача — бинарная классификация с **умеренным дисбалансом**: положительный
класс (отток) занимает около четверти выборки, поэтому accuracy бесполезна как
метрика, а при обучении понадобится взвешивание/подбор порога.

In [ ]:
counts = clean[TARGET].value_counts().sort_index()
rate = clean[TARGET].mean()
print(counts.to_string())
print(f"\nДоля оттока (churn=1): {rate:.1%}")
print(f"Дисбаланс классов ~ {counts[0] / counts[1]:.1f} : 1 (остался : ушёл)")

fig, ax = plt.subplots(figsize=(4.5, 3))
ax.bar(["Остался", "Ушёл"], counts.values, color=[BLUE, ORANGE], width=0.6)
style_ax(ax, "y")
for x, v in enumerate(counts.values):
    ax.annotate(f"{v}\n({v / counts.sum():.0%})", (x, v), ha="center", va="bottom",
                fontsize=9, color=INK, xytext=(0, 2), textcoords="offset points")
ax.set_ylim(0, counts.max() * 1.18)
ax.set_ylabel("Число клиентов")
ax.set_title("Распределение целевой переменной")
plt.show()

## 3. Типы, пропуски, дубликаты

После очистки в `data.py` пропусков нет. Единственная нетривиальная правка:
`Total Charges` в сыром файле хранился **как текст** и был пустым у 11 клиентов
с нулевым стажем (`tenure = 0`, ещё не выставлен первый счёт) — это уже
приведено к числу и заполнено нулём. Полные дубли строк оставлены как есть:
это неразличимые по признакам клиенты, а не ошибка загрузки.

In [ ]:
info = pd.DataFrame({
    "dtype": clean.dtypes.astype(str),
    "n_unique": clean.nunique(),
    "n_missing": clean.isna().sum(),
})
print(info.to_string())
print("\nПропусков всего:", int(clean.isna().sum().sum()))
print("Полных дубликатов строк:", int(clean.duplicated().sum()))

## 4. Анализ ликажа (ключевая секция)

Сырой IBM-сэмпл содержит колонки, которые **знают ответ** или недоступны в
момент скоринга живого клиента. Их присутствие завысило бы метрики и сделало
модель бесполезной в проде. Проверяем три главных подозреваемых на сыром файле.

In [ ]:
raw = pd.read_excel(ROOT / "data" / "raw" / "Telco_customer_churn.xlsx")
y_raw = raw["Churn Value"].astype(int)

# (1) Churn Reason — заполняется только при уходе
n_reason_missing = int(raw["Churn Reason"].isna().sum())
n_nonchurn = int((y_raw == 0).sum())

# (2) Churn Score — насколько сильно предсказывает таргет
auc_score = roc_auc_score(y_raw, raw["Churn Score"])

# (3) CLTV — сравнение распределений по классам
cltv = raw.groupby(y_raw)["CLTV"].agg(["mean", "median", "std"]).round(1)
cltv.index = ["остался", "ушёл"]

print(f"[Churn Reason] пропусков: {n_reason_missing}  |  неушедших клиентов: {n_nonchurn}"
      f"  ->  совпадает: {n_reason_missing == n_nonchurn}")
print(f"[Churn Score ] ROC-AUC против churn: {auc_score:.3f}  (почти идеальное разделение)")
print(f"[CLTV        ] распределение по классам:\n{cltv.to_string()}")
print(f"\nLEAKY_COLUMNS (из churn.data): {LEAKY_COLUMNS}")

**Вывод — что и почему удалено (см. `churn.data`):**

- **`Churn Reason`** — число пропусков (**5174**) в точности равно числу
  неушедших клиентов: поле заполняется *только когда клиент уже ушёл* →
  **чистый ликаж**, в момент скоринга его не существует.
- **`Churn Score`** — ROC-AUC этого поля против таргета **≈ 0.94**: это выход
  **чужой (IBM) модели, обученной на том же ответе**. Использовать его — значит
  учить модель списывать у другой модели, а не у данных.
- **`CLTV`** — расчётная «пожизненная ценность» с непрозрачным происхождением;
  распределения по классам различаются слабо, но сам способ расчёта неизвестен и
  мог включать факт оттока → **выброшен консервативно**.
- **`Churn Label`** — строковый дубликат таргета `Churn Value`.
- **`Count / Country / State`** — константы; **`CustomerID`** — идентификатор;
  **`City / Zip / Lat Long / Latitude / Longitude`** — гео-шум (датасет из одного
  штата, высокая кардинальность на 7k строк).

Итого модель обучается только на признаках, **известных до факта оттока**.

## 5. Разведочные графики

Единая палитра: основной цвет — синий `#2563eb`, акцент (отток) — оранжевый
`#ea580c`, контекст — серый `#6b7280`. Все графики сохраняются в
`reports/figures/eda_*.png` (dpi 150).

### 5.1 Отток по типу контракта

In [ ]:
s = clean.groupby("contract")[TARGET].mean().mul(100).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(s.index, s.values, color=BLUE, height=0.6)
style_ax(ax, "x")
label_bars(ax, "h")
ax.set_xlim(0, s.max() * 1.18)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.set_xlabel("Доля оттока, %")
ax.set_title("Отток по типу контракта")
save(fig, "eda_churn_by_contract.png")
plt.show()

### 5.2 Отток по стажу (tenure)

In [ ]:
bins = [0, 6, 12, 24, 48, np.inf]
labels = ["0–6", "6–12", "12–24", "24–48", "48+"]
tb = pd.cut(clean["tenure_months"], bins=bins, labels=labels, right=False)
s = clean.groupby(tb, observed=True)[TARGET].mean().mul(100).reindex(labels)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(s.index, s.values, color=BLUE, width=0.65)
style_ax(ax, "y")
label_bars(ax, "v")
ax.set_ylim(0, s.max() * 1.15)
ax.set_ylabel("Доля оттока, %")
ax.set_xlabel("Стаж клиента, месяцев")
ax.set_title("Отток по стажу: новички уходят кратно чаще")
save(fig, "eda_churn_by_tenure.png")
plt.show()

### 5.3 Отток по типу интернета и способу оплаты

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, col, title in [(axes[0], "internet_service", "Тип интернета"),
                       (axes[1], "payment_method", "Способ оплаты")]:
    s = clean.groupby(col)[TARGET].mean().mul(100).sort_values()
    ax.barh(s.index, s.values, color=BLUE, height=0.6)
    style_ax(ax, "x")
    label_bars(ax, "h")
    ax.set_xlim(0, s.max() * 1.30)
    ax.set_xlabel("Доля оттока, %")
    ax.set_title(title)
fig.suptitle("Отток по интернет-услуге и способу оплаты", fontweight="bold", y=1.03)
fig.tight_layout()
save(fig, "eda_churn_by_internet_payment.png")
plt.show()

### 5.4 Распределение месячного платежа по классам

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(data=clean[clean[TARGET] == 0], x="monthly_charges", fill=True,
            alpha=0.35, color=BLUE, linewidth=1.6, label="Остался", ax=ax)
sns.kdeplot(data=clean[clean[TARGET] == 1], x="monthly_charges", fill=True,
            alpha=0.35, color=ORANGE, linewidth=1.6, label="Ушёл", ax=ax)
style_ax(ax, "y")
ax.set_xlabel("Месячный платёж, $")
ax.set_ylabel("Плотность")
ax.set_title("Месячный платёж: ушедшие платят заметно больше")
ax.legend(frameon=False)
save(fig, "eda_monthly_charges_by_churn.png")
plt.show()

### 5.5 Отток по числу подключённых услуг (после build_features)

In [ ]:
s = feat.groupby("num_services")[TARGET].mean().mul(100)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(s.index.astype(int), s.values, color=BLUE, width=0.7)
style_ax(ax, "y")
label_bars(ax, "v")
ax.set_ylim(0, s.max() * 1.15)
ax.set_xticks(s.index.astype(int))
ax.set_xlabel("Число подключённых услуг (num_services)")
ax.set_ylabel("Доля оттока, %")
ax.set_title("Отток по числу подключённых услуг")
save(fig, "eda_churn_by_num_services.png")
plt.show()

### 5.6 Сила связи признаков с оттоком

Для **числовых** признаков — point-biserial (корреляция Пирсона с бинарным
таргетом, со знаком). Для **категориальных** — корреляция доли оттока в
категории с таргетом (по построению неотрицательна). Оранжевый — признак
повышает риск оттока, синий — снижает.

In [ ]:
corr = {}
for f in NUMERIC_FEATURES:
    corr[f] = np.corrcoef(feat[f], feat[TARGET])[0, 1]
for f in CATEGORICAL_FEATURES:
    enc = feat.groupby(f)[TARGET].transform("mean")
    corr[f] = np.corrcoef(enc, feat[TARGET])[0, 1]
cs = pd.Series(corr)
top = cs.reindex(cs.abs().sort_values(ascending=False).index).head(12).iloc[::-1]

colors = [ORANGE if v >= 0 else BLUE for v in top.values]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top.index, top.values, color=colors, height=0.66)
style_ax(ax, "x")
for p in ax.patches:
    v = p.get_width()
    ax.annotate(f"{v:+.2f}", (v, p.get_y() + p.get_height() / 2),
                ha="left" if v >= 0 else "right", va="center", fontsize=8.5, color=INK,
                xytext=(3 if v >= 0 else -3, 0), textcoords="offset points")
ax.axvline(0, color=GRAY, linewidth=0.8)
lim = max(abs(top.min()), abs(top.max())) * 1.28
ax.set_xlim(-lim, lim)
ax.set_xlabel("Связь с оттоком (числовые — point-biserial, категории — corr доли оттока)")
ax.set_title("Топ-12 признаков по силе связи с оттоком")
save(fig, "eda_target_correlations.png")
plt.show()

## 6. Sanity-check важностей (LightGBM)

Быстрая проверка, что в топе важностей стоит **осмысленная бизнес-логика**, а не
случайно просочившийся ликаж. Модель — LightGBM с дефолтными параметрами
(300 деревьев) на `train.parquet`; важности типа **gain** агрегированы обратно к
исходным признакам (после OneHot).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMClassifier

train = pd.read_parquet(ROOT / "data" / "processed" / "train.parquet")
X, y = train[ALL_FEATURES], train[TARGET]

pre = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
     ("num", "passthrough", NUMERIC_FEATURES)],
    verbose_feature_names_out=False,
)
Xt = pre.fit_transform(X)
names = pre.get_feature_names_out()

clf = LGBMClassifier(n_estimators=300, random_state=RANDOM_STATE, verbose=-1)
clf.fit(Xt, y)
gain = pd.Series(clf.booster_.feature_importance(importance_type="gain"), index=names)


def source_of(name):
    if name in NUMERIC_FEATURES:
        return name
    cands = [f for f in CATEGORICAL_FEATURES if name.startswith(f + "_")]
    return max(cands, key=len) if cands else name


share = gain.groupby(gain.index.map(source_of)).sum().sort_values(ascending=False)
share = share / share.sum() * 100
top10 = share.head(10).iloc[::-1]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top10.index, top10.values, color=BLUE, height=0.66)
style_ax(ax, "x")
label_bars(ax, "h", fmt="{:.1f}%")
ax.set_xlim(0, top10.max() * 1.18)
ax.set_xlabel("Вклад в прирост (gain), %")
ax.set_title("LightGBM: топ-10 признаков по важности (gain)")
save(fig, "eda_lgbm_importance.png")
plt.show()

print(share.head(10).round(1).to_string())

**Вывод.** В топе — `contract`, `tenure_months`, `monthly_charges` и их
производные (`charge_diff`, `avg_service_cost`, `total_charges`). Ни одна фича не
берёт на себя *подозрительно* большую долю (лидер `contract` ≈ 22 % gain — это
нормально: тип контракта действительно ключевой драйвер удержания). Важность
распределена по осмысленным бизнес-факторам, а не сконцентрирована в одном
«всезнающем» признаке — **признаков ликажа нет**.

## 7. Выводы — 5 гипотез с числами (для README)

1. **Тип контракта — драйвер №1.** На тарифе *month-to-month* уходят **42.7 %**
   клиентов против **2.8 %** на *two-year* — в **≈ 15 раз** чаще. Перевод на
   длинные контракты — самый сильный рычаг удержания.
2. **Первые месяцы решают всё.** В когорте *0–6 мес.* отток **54.3 %**, в
   когорте *48+ мес.* — **9.6 %** (≈ в **5.6 раза** ниже). Онбординг и ранние
   касания критичны.
3. **Способ оплаты — маркер риска.** *Electronic check* — **45.3 %** оттока
   против **15.2 %** у автосписания с карты (**≈ в 3 раза** выше); ручные платежи
   сигналят о слабой привязке к сервису.
4. **Fiber optic уходит чаще.** Оптика — **41.9 %** оттока против **19.0 %** у DSL
   и **7.4 %** у клиентов без интернета: премиальный, но нелояльный сегмент
   (вероятно, цена/качество).
5. **Высокий чек = выше риск.** Медианный месячный платёж у ушедших **$79.6**
   против **$64.4** у оставшихся (**+$15/мес**); отдельно уязвимы пенсионеры —
   **41.7 %** оттока против **23.6 %** у остальных.